# Smoke Test

Run all cells top-to-bottom. If everything prints **OK**, your environment is ready for the exercises.

Before you run: **Select Kernel** (top-right) → *Python Environments…* → pick the entry ending in `streaming/.venv/bin/python`.

## 1. Imports

Proves the venv is wired up and `confluent_kafka` is installed.

In [ ]:
from confluent_kafka import Producer, Consumer
from confluent_kafka.admin import AdminClient
import json, time

print('OK — confluent_kafka imported')

## 2. Connect to Redpanda and list topics

Reaches the broker on the Docker-internal address `redpanda:29092`. Should print at least the empty list (or whatever topics you've already created).

In [ ]:
BROKER = 'redpanda:29092'

admin = AdminClient({'bootstrap.servers': BROKER})
topics = admin.list_topics(timeout=5).topics
print(f'OK — connected to {BROKER}, {len(topics)} topic(s) so far: {sorted(topics)}')

## 3. Create the topics for the exercises

The exercises use two topics of a food-delivery app:

- `rider-locations` — GPS pings of the riders (where is each rider right now?)
- `orders` — order events (Anna orders a pizza, the restaurant accepts, …)

Redpanda would create a topic automatically on the first write, but only with **one** partition. With one partition every event lands on partition 0, and you could not see how keys are routed. So we create both topics with **3 partitions**. Running this cell again is harmless.

In [ ]:
from confluent_kafka.admin import AdminClient, NewTopic, NewPartitions

PARTITIONS = 3
admin    = AdminClient({'bootstrap.servers': BROKER})
existing = admin.list_topics(timeout=5).topics

for name in ['rider-locations', 'orders']:
    if name not in existing:
        admin.create_topics([NewTopic(name, num_partitions=PARTITIONS)])[name].result()
        print(f'OK — created {name!r} with {PARTITIONS} partitions')
    elif len(existing[name].partitions) < PARTITIONS:
        admin.create_partitions([NewPartitions(name, PARTITIONS)])[name].result()
        print(f'OK — {name!r} grown to {PARTITIONS} partitions')
    else:
        print(f'OK — {name!r} already has {len(existing[name].partitions)} partitions')

## 4. Round-trip: produce one event and read it back

Writes a single message to a `smoke-test` topic and immediately consumes it.

In [ ]:
TOPIC = 'smoke-test'

producer = Producer({'bootstrap.servers': BROKER})
producer.produce(TOPIC, key=b'hello', value=json.dumps({'ts': time.time()}).encode())
producer.flush()
print(f'OK — produced 1 event to {TOPIC!r}')

consumer = Consumer({
    'bootstrap.servers': BROKER,
    'group.id':          f'smoke-{int(time.time())}',  # fresh group -> reads from earliest
    'auto.offset.reset': 'earliest',
})
consumer.subscribe([TOPIC])

for _ in range(10):              # poll up to ~5 seconds for a message
    msg = consumer.poll(0.5)
    if msg and not msg.error():
        print(f'OK — read back: key={msg.key()} value={msg.value()}')
        break
else:
    print('FAIL — no message received within 5 s')

consumer.close()

If all four sections printed **OK**, you're done — head to [`../02_demo/demo_produce.ipynb`](../02_demo/demo_produce.ipynb) and then [`../03_exercise/`](../03_exercise/).